#### This notebook is to help clean and transform the 2k26 roster attributes and data to match the other roster years

##### End Schema:
```json
{
        "name": "FirstName LastName",
        "team": "TeamName",
        "overallAttribute":
        "closeShot":
        "midRangeShot":
        "threePointShot":
        "freeThrow":
        "shotIQ":
        "offensiveConsistency":
        "layup":
        "standingDunk":
        "drivingDunk":
        "postHook":
        "postFade":
        "postControl":
        "drawFoul":
        "hands":
        "interiorDefense":
        "perimeterDefense":
        "steal":
        "block":
        "helpDefenseIQ":
        "passPerception":
        "defensiveConsistency":
        "speed":
        "agility":
        "strength":
        "vertical":
        "stamina":
        "hustle":
        "overallDurability":
        "passAccuracy":
        "ballHandle":
        "speedWithBall":
        "passIQ":
        "passVision":
        "offensiveRebound":
        "defensiveRebound":
    }

##### Imports

In [59]:
import json

##### Load In the Raw Data

In [64]:
with open("2k26_roster_raw.json", "r") as f:
    data = json.load(f)

In [65]:
print(json.dumps(data, indent=4))

[
    {
        "_creationTime": 1763351341164.1975,
        "_id": "j97dy9mky9kendg9sp96bwazds7vj0y9",
        "archetype": "Triple-Double Threat",
        "attributes": {
            "agility": 62,
            "ballHandle": 76,
            "block": 61,
            "closeShot": 99,
            "defensiveConsistency": 80,
            "defensiveRebound": 97,
            "drawFoul": 98,
            "drivingDunk": 75,
            "drivingLayup": 91,
            "durability": 83,
            "freeThrow": 83,
            "hands": 90,
            "helpDefenseIQ": 83,
            "hustle": 80,
            "interiorDefense": 83,
            "midRangeShot": 98,
            "offensiveConsistency": 98,
            "offensiveRebound": 73,
            "passAccuracy": 93,
            "passIQ": 98,
            "passPerception": 82,
            "passVision": 99,
            "perimeterDefense": 61,
            "postControl": 98,
            "postFade": 95,
            "postHook": 94,
            "shotI

##### Deleting the unecessary keys

In [68]:
keys_to_remove = [
    "_creationTime",
    "_id",
    "archetype",
    "badges",
    "build",
    "createdAt",
    "gameVersion",
    "height",
    "playerImage",
    "playerUrl",
    "positions",
    "ratingHistory",
    "slug",
    "team",
    "teamImg",
    "teamType",
    "weight",
    "wingspan",
]

for player in data:
    for key in keys_to_remove:
        player.pop(key, None)

##### Some players have multiple versions, so need to delete the older player

###### This will be done by keeping the last updated attribute instead of straight up deleting it, then we will add the player to a set and if they already exist in the set, we note that down as a repeat.

In [87]:
existing_names = set()
duplicates = set()
for player in data:
    if player['name'] in existing_names:
        duplicates.add(player['name'])
    else:
        existing_names.add(player['name'])

In [88]:
print(f"Found {existing_names} duplicates:")

Found {'Keon Ellis', 'Adou Thiero', 'Bones Hyland', 'Bobby Portis Jr.', 'Neemias Queta', 'John Konchar', 'Drake Powell', 'John Poulakidas', 'De’Anthony Melton', 'Jordan Miller', 'Gary Payton II', 'Pacome Dadiet', 'Aaron Gordon', 'PJ Hall', 'Cam Spencer', 'Mac McClung', 'Anfernee Simons', 'Jaren Jackson Jr.', 'Will Richard', 'Johnny Juzang', 'Jalen Pickett', 'Dalton Knecht', 'Mark Sears', 'Derik Queen', 'Damian Lillard', 'Tyus Jones', 'Kon Knueppel', 'Herbert Jones', 'Adem Bona', 'Jordan Goodwin', 'LaMelo Ball', 'Ayo Dosunmu', 'Rob Dillingham', 'Dillon Brooks', 'Riley Minix', 'Draymond Green', 'Gabe Vincent', 'Dean Wade', 'Ziaire Williams', 'Thomas Bryant', 'Kevin Love', 'Doug McDermott', 'Tristen Newton', 'Emanuel Miller', 'Jeff Green', 'Nick Smith Jr.', 'Nikola Djurisic', 'Tyrese Proctor', 'John Tonje', 'Jeremiah Fears', 'Chris Livingston', 'Kel’el Ware', 'Adama Bal', 'Jusuf Nurkic', 'Jeremiah Robinson-Earl', 'Ryan Dunn', 'Julian Strawther', 'Dwight Powell', 'Tolu Smith', 'Max Shulga'

###### We will compare the player cards and see which one is the older version based on the "lastUpdated" which shouldn't be before 2026-05 or that one will be the one getting deleted

In [89]:
for player in data:
    # player last updated before may 2026
    if player['name'] in duplicates and player['lastUpdated'] < "2026-05-01T00:00:00Z":
        data.remove(player)

##### Sanity Check

In [92]:
players = []
for player in data:
    players.append(player['name'])

if len(players) != len(set(players)):
    print("Duplicates still exist!")


##### Removing lastUpdated

In [95]:
for player in data:
    player.pop("lastUpdated", None)

##### Transforming the dataset into a similar format as previous years

In [ ]:
# Flatten Attributes into the main dict
for player in data:
    attributes = player.pop("attributes", {})
    for key, value in attributes.items():
        player[key] = value

In [100]:
# Rename "overall" to "overallAttribute", "drivingLayup" to "layup" and "durability" to "overallDurability"
for player in data:
    player["overallAttribute"] = player.pop("overall", None)
    player["layup"] = player.pop("drivingLayup", None)
    player["overallDurability"] = player.pop("durability", None)

##### Dumping Data Into 2k26_roster.json file

In [99]:
with open("2k26_roster.json", "w") as f:
    json.dump(data, f, indent=4)